## Overview
In this comprehensive capstone project, you'll analyze TrendWave Media's social media engagement data to derive actionable insights. You'll apply the full range of data science skills you've learned throughout the course while leveraging AI tools to enhance your workflow.

## Learning Outcomes
- Execute a complete data science workflow from ingestion to reporting
- Apply advanced data cleaning and EDA techniques
- Create insightful visualizations
- Perform statistical analysis
- Document your process using Git

## Dataset Information
You'll be working with the <b>Capstone_Project_synthetic_social_media_data.csv</b> dataset, which contains social media engagement metrics including:
- Post metadata (ID, timestamp, content)
- User demographics
- Engagement metrics (likes, shares, comments)
- Sentiment scores
- Topic classifications

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from pathlib import Path

## Activities 
### Activity 1: Data Exploration and Preparation 
Let's start by understanding our user engagement data.

<b>Step 1:</b> Adjust paths to import data

In [ ]:
BASE_DIR = Path.cwd().resolve().parents[1]
DATA_DIR = BASE_DIR / "data"
user_engagement_path = DATA_DIR / 'Capstone_Project_synthetic_social_media_data.csv'

In [ ]:
df = pd.read_csv(user_engagement_path)
df.head()

<b>Step 2:</b> Data Quality Assessment

In [ ]:
print(f'unique ids: {df.PostID.nunique()}')
print(f'unique users: {df.UserID.nunique()}')

In [ ]:
def show_basic_df_info(df: pd.DataFrame) -> None:
    """
    Display basic information and summary statistics of the DataFrame.

    Args:
        df (pd.DataFrame): The DataFrame to analyze.

    Prints:
        - DataFrame info including data types and memory usage.
        - Count of missing values per column.
        - Summary statistics for numeric columns.
    """
    print("Dataset Info:")
    display(df.info())

    print("\nMissing Values:")
    display(df.isna().sum())

    print("\nBasic Statistics:")
    print(df.describe())


def get_categorical_counts(df: pd.DataFrame, cat_cols: list[str] = []) -> None:
    """
    Print value counts for specified or inferred categorical columns.

    Args:
        df (pd.DataFrame): The DataFrame containing categorical data.
        cat_cols (list[str], optional): List of categorical column names. 
            If empty, all object-type columns are used.

    Prints:
        Value counts for each categorical column.
    """
    if len(cat_cols) == 0:
        cat_cols = df.select_dtypes(include=['object']).columns
    for col in cat_cols:
        print(f"\n{col} distribution:")
        print(df[col].value_counts())

In [ ]:
show_basic_df_info(df)

In [ ]:
get_categorical_counts(df, cat_cols=['PostContent', 'Hashtags', 'PostTopic', 'UserLocation', 'UserGender'])

<b>Step 3:</b> Data Cleaning

In [ ]:
# Handle missing values and standardize formats


def convert_dates(df: pd.DataFrame, date_cols: list[str]) -> pd.DataFrame:
    """
    Convert specified columns in the DataFrame to datetime format.

    Args:
        df (pd.DataFrame): The input DataFrame containing the data.
        date_cols (list[str]): List of column names to convert to datetime.

    Returns:
        pd.DataFrame: A DataFrame with the specified columns converted to datetime.

    Notes:
        The datetime conversion uses format='mixed' to handle multiple date formats.
    """
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], format='mixed')
    return df



def impute_nulls(df: pd.DataFrame) -> pd.DataFrame:
    """
    Impute missing values in specific columns of the DataFrame.

    Args:
        df (pd.DataFrame): The input DataFrame with potential null values.

    Returns:
        pd.DataFrame: A DataFrame with missing values imputed.

    Notes:
        'SentimentScore' is imputed with the median value.
        'UserGender' is imputed with the mode (most frequent) value.
    """
    df['SentimentScore'] = df['SentimentScore'].fillna(df['SentimentScore'].median())
    df['UserGender'] = df['UserGender'].fillna(df['UserGender'].mode().iloc[0])
    return df


def standardize_strs(df: pd.DataFrame, str_cols: list[str]) -> pd.DataFrame:
    """
    Standardize string columns by stripping whitespace and converting to lowercase.

    Args:
        df (pd.DataFrame): The input DataFrame containing string columns.
        str_cols (list[str]): List of column names to standardize.

    Returns:
        pd.DataFrame: A DataFrame with standardized string columns.
    """
    for col in str_cols:
        df[col] = df[col].str.strip().str.lower()
    return df


def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean the DataFrame by converting dates, imputing nulls, and standardizing strings.

    Args:
        df (pd.DataFrame): The input DataFrame to clean.

    Returns:
        pd.DataFrame: A cleaned copy of the DataFrame with processed columns.

    Notes:
        Applies convert_dates to 'PostDateTime'.
        Applies impute_nulls to fill missing values.
        Applies standardize_strs to specified string columns.
    """
    df_clean = df.copy()
    df_clean = convert_dates(df_clean, date_cols=['PostDateTime'])
    df_clean = impute_nulls(df_clean)
    df_clean = standardize_strs(df_clean, str_cols=['UserGender', 'UserLocation', 'PostTopic', 'Hashtags', 'PostContent'])
    return df_clean

In [ ]:
df_clean = clean_data(df)

In [ ]:
show_basic_df_info(df_clean)

### Activity 2: Exploratory Data Analysis
With clean data in hand, TrendWave's content strategy team needs to understand what drives user engagement across different content types and topics. They want insights that can directly inform content creation and posting strategies.

<b>Step 1:</b> Basic Statistical Analysis

In [ ]:
def get_engagement_sum(df: pd.DataFrame) -> dict[str, any]:
    """
    Calculate total engagement metrics from the DataFrame.

    Args:
        df (pd.DataFrame): Cleaned DataFrame with 'Likes', 'Comments', and 'Shares' columns.

    Returns:
        dict[str, any]: Dictionary with total counts for 'total_likes', 'total_comments', and 'total_shares'.
    """
    results = {}
    results['total_likes'] = df['Likes'].sum()
    results['total_comments'] = df['Comments'].sum()
    results['total_shares'] = df['Shares'].sum()
    return results


def get_engagement_by_topic(df: pd.DataFrame) -> dict[str, any]:
    """
    Compute engagement statistics grouped by post topic.

    Args:
        df (pd.DataFrame): Cleaned DataFrame with 'PostTopic', 'Likes', 'Comments', and 'Shares' columns.

    Returns:
        dict[str, any]: Dictionary with a DataFrame under 'topic_engagement' containing mean, min, and max of engagement metrics per topic.
    """
    results = df.groupby('PostTopic').agg({
        'Likes': ['mean', 'min', 'max'],
        'Comments': ['mean', 'min', 'max'],
        'Shares': ['mean', 'min', 'max']
    }).round(2)
    
    return results.to_dict(orient='index')


def get_engagement_by_time(df: pd.DataFrame) -> dict[str, any]:
    """
    Compute engagement statistics grouped by hour of day and day of week.

    Args:
        df (pd.DataFrame): Cleaned DataFrame with 'PostDateTime', 'Likes', 'Comments', and 'Shares' columns.

    Returns:
        dict[str, any]: Dictionary with DataFrames under 'hourly_engagement' and 'day_of_week_engagement' keys containing mean, min, and max of engagement metrics.
    """
    ret = {}
    df['hour'] = df['PostDateTime'].dt.hour
    df['day_of_week'] = df['PostDateTime'].dt.dayofweek
    results = df.groupby('hour').agg({
        'Likes': ['mean', 'min', 'max'],
        'Comments': ['mean', 'min', 'max'],
        'Shares': ['mean', 'min', 'max']
    })
    ret = results.round(2).to_dict()
    results = df.groupby('day_of_week').agg({
        'Likes': ['mean', 'min', 'max'],
        'Comments': ['mean', 'min', 'max'],
        'Shares': ['mean', 'min', 'max']
    }).round(2)
    
    return {**ret, **results.to_dict(orient='index')}


def get_engagement_metrics(df: pd.DataFrame) -> dict[str, any]:
    """
    Aggregate various engagement metrics into a single dictionary.

    Args:
        df (pd.DataFrame): Cleaned DataFrame suitable for engagement analysis.

    Returns:
        dict[str, any]: Combined dictionary of total engagement, topic-based, and time-based engagement metrics.
    """
    engagement_sum = get_engagement_sum(df)
    engagement_by_topic = get_engagement_by_topic(df)
    engagement_by_time = get_engagement_by_time(df)
    
    return {**engagement_sum, **engagement_by_topic, **engagement_by_time}

In [ ]:
engagement_metrics = get_engagement_metrics(df_clean)
engagement_metrics

<b>Step 3:</b> Visualization Creation

In [ ]:
# - A boxplot for Topic engagement distribution
plt.figure(figsize=(12, 6))
sns.boxplot(x='PostTopic', y='Likes', data=df_clean)
plt.title('Likes by Topic')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Time series of engagement
daily_engagement = df_clean.groupby(df_clean['PostDateTime'].dt.date)['Likes'].mean()
plt.figure(figsize=(12, 6))
daily_engagement.plot()
plt.title('Daily Average Likes')
plt.tight_layout()
plt.show()

In [ ]:
# Interactive scatter plot
fig = px.scatter(
    df_clean, 
     x='Likes', 
     y='Shares',
     color='PostTopic',
     hover_data=['Comments'],
     title='Engagement Relationships'
)
fig.show()

### Activity 3: Statistical Analysis and Documentation
The executive team needs concrete evidence to support strategic decisions about content direction and resource allocation. The hypothesis being tested is whether there is an association between post topics and user demographics, which could influence content performance. Your analysis will help validate or challenge existing assumptions about content performance.

<b>Step 1:</b> Hypothesis Testing

In [ ]:
# Perform Chi-Squared test on the relationship between PostTopic and UserGender 
contingency = pd.crosstab(df['PostTopic'], df['UserGender'])
contingency

In [ ]:
def chi2_test(contingency_df: pd.DataFrame) -> dict[str, any]:
    """
    Perform a Chi-square test of independence on a contingency table.

    Args:
        contingency_df (pd.DataFrame): Contingency table with observed frequencies.

    Returns:
        dict[str, any]: Dictionary containing:
            - 'chi2_statistic' (float): The Chi-square test statistic.
            - 'p_value' (float): The p-value of the test.
            - 'degrees_of_freedom' (int): Degrees of freedom for the test.
    """
    stat, p_value, dof, expected = stats.chi2_contingency(contingency_df)
    return {
        'chi2_statistic': stat,
        'p_value': p_value,
        'degrees_of_freedom': dof
    }


def one_way_anova_test_topic(df: pd.DataFrame, var: str) -> dict[str, any]:
    """
    Perform a one-way ANOVA test to compare means of a variable across post topics.

    Args:
        df (pd.DataFrame): DataFrame containing the data with a 'PostTopic' column.
        var (str): The name of the numeric variable to test.

    Returns:
        dict[str, any]: Dictionary containing:
            - 'f_statistic' (float): The ANOVA F-statistic.
            - 'p_value' (float): The p-value of the test.
    """
    f_stat, p_value = stats.f_oneway(*[group[var] for name, group in df.groupby('PostTopic')])
    return {
        'f_statistic': f_stat,
        'p_value': p_value
    }

In [ ]:
print(chi2_test(contingency_df=contingency))

In [ ]:
print(one_way_anova_test_topic(df=df_clean, var='Likes'))

In [ ]:
print(one_way_anova_test_topic(df=df_clean, var='Comments'))

In [ ]:
print(one_way_anova_test_topic(df=df_clean, var='Shares'))

# Statistical tests and interpretation

## Chi-square Test Results

- **Chi2 statistic:** ≈ 7.75
- **p-value:** ≈ 0.458
- **Degrees of freedom:** 8

Since the p-value (0.458) is much greater than the common significance level of 0.05, there is not enough evidence to reject the null hypothesis. The null hypothesis states that **UserGender and PostTopic are independent**, meaning gender does not significantly affect the distribution of posts across topics.

**Conclusion:** Gender does not appear to influence the amount of posts per topic in the data.

---

## One-way ANOVA Results (Likes per Topic)

- **F-statistic:** ≈ 0.38
- **p-value:** ≈ 0.83

## One-way ANOVA Results (Comments per Topic)

- **F-statistic:** ≈ 1.015
- **p-value:** ≈ 0.3978

## One-way ANOVA Results (Comments per Topic)

- **F-statistic:** ≈ 1.4370
- **p-value:** ≈ 0.218

Since the p-values are much higher than 0.05, we fail to reject the null hypothesis that the mean likes, comments and shares are the same across all topics.

**Conclusion:** There is no statistically significant difference in average likes, comments and shares between the different post topics.